<a href="https://colab.research.google.com/github/halizahhhh/Tugas-Akhir-Siti-Nurhalizah/blob/main/Notebook_Analisis_Sentimen_IndoBERT_vs_TF_IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Perbandingan Kinerja IndoBERT dan TF-IDF dalam Mengklasifikasikan Sentimen Evaluasi Dosen Oleh Mahasiswa (EDOM) Menggunakan Algoritma K-Nearest Neighbor**

Notebook ini digunakan untuk melakukan proses analisis dan eksperimen dalam penelitian yang berfokus pada perbandingan dua teknik representasi teks, yaitu IndoBERT dan TF-IDF, dalam tugas klasifikasi sentimen komentar EDOM.

Tujuan Penelitian :
1. Membandingkan kinerja representasi teks IndoBERT dan TF-IDF dalam mengklasifikasikan sentimen pada data Evaluasi Dosen oleh Mahasiswa (EDOM).
2. Menentukan model klasifikasi sentimen yang paling optimal, dengan membandingkan kombinasi IndoBERT + KNN versus TF-IDF + KNN.



---



# **Import Library**

In [ ]:
# =============================
# 0. IMPORT & SETUP
# =============================
# Install (jika belum terpasang) jalankan di notebook:
!pip install transformers torch sastrawi imbalanced-learn seaborn google-generativeai tqdm

import os
import time
import re
import string
import json
from io import StringIO
from tqdm import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, precision_score, recall_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Sastrawi utilities
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("Libraries loaded. Torch device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# **1. Load Dataset**

In [ ]:
df = pd.read_excel('link_data')
print("Total data:", len(df))

pd.set_option('display.max_rows', 10)  # hanya menampilkan sebagian (awal + akhir)
df

In [ ]:
df.info()

# **2. Data Cleaning**

In [ ]:
# 2.1 Hapus duplikat berdasarkan kolom 'Komentar'
print("Jumlah duplikat komentar awal:", df['Komentar'].duplicated().sum())
df = df.drop_duplicates(subset='Komentar').reset_index(drop=True)
print("Setelah hapus duplikat:", len(df))

In [ ]:
# 2.2 Hapus komentar tidak bermakna (fungsi kamu, dipertahankan logikanya)
def remove_undefined_text(text):
    if pd.isna(text):
        return None
    t = str(text).strip().lower()
    invalid = [
        '-', '.', ',', '..', '...', '', ' ','"."', '_', '__',
        'tidak ada', 'cukup', 'tidak', 'kosong', 'tidaj ada'
    ]
    if len(t) <= 3:
        return None
    if len(t.split()) <= 3 and t in invalid:
        return None
    return text

df['Komentar'] = df['Komentar'].apply(remove_undefined_text)
deleted_rows = df[df['Komentar'].isna()]
print("Jumlah data tidak bermakna yang dihapus:", len(deleted_rows))
df = df.dropna(subset=['Komentar']).reset_index(drop=True)
print("Jumlah data akhir setelah bersih:", len(df))

In [ ]:
# 2.3 Hilangkan whitespace berlebih
df['Komentar'] = df['Komentar'].astype(str).str.strip()
df['Komentar'] = df['Komentar'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
# Tampilkan beberapa contoh
display(df.head(5))

# **3. Preprocessing Teks**

In [ ]:
# 3.1 Kamus normalisasi
normalisasi = {
    "gk": "tidak", "ga": "tidak", "nggak": "tidak", "bgt": "banget", "bngt": "banget", "bener": "benar", "dosenny": "dosen nya", "mahasiswanya": "mahasiswa nya",
    "bgs": "bagus", "bagusss": "bagus", "baguss": "bagus", "bkin": "bikin", "bgin": "bikin", "bbrp": "beberapa", "dosenx": "dosen", "dosenya": "dosen", "mhs": "mahasiswa",
    "mhsw": "mahasiswa", "bnyk": "banyak", "bnk": "banyak", "banyakkk": "banyak", "krn": "karena", "krena": "karena", "karna": "karena", "jg": "juga", "jga": "juga", "jugaaa": "juga",
    "sbnrnya": "sebenarnya", "sbnr": "sebenarnya", "brrti": "berarti", "udh": "sudah", "udah": "sudah", "sdh": "sudah", "blm": "belum", "belomm": "belum",
    "kl": "kalau", "klo": "kalau", "klu": "kalau", "kalo": "kalau", "aj": "saja", "aja": "saja", "sj": "saja", "mksd": "maksud", "mksud": "maksud",
    "ngajar": "mengajar", "ngasih": "memberikan", "ngambil": "mengambil", "njelasin": "menjelaskan", "jelasin": "menjelaskan", "mantep": "mantap", "mantab": "mantap",
    "keren banget": "sangat baik", "baik banget": "sangat baik", "ok": "oke", "oke banget": "sangat baik", "lumayan kok": "cukup baik", "cukup baik lah": "cukup baik",
    "td": "tadi", "skr": "sekarang", "nnti": "nanti", "materinya": "materi", "penjelasannya": "penjelasan", "pembelajarannya": "pembelajaran", "tp": "tapi", "tpi": "tapi"
}

def normalize_text(text):
    text = text.lower()
    for key, value in normalisasi.items():
        text = re.sub(r"\b{}\b".format(re.escape(key)), value, text)
    return text

In [ ]:
# 3.2 Cleaning teks

def clean_text(text):
    # Case folding
    text = str(text).lower()
    # Hilangkan angka
    text = re.sub(r'\d+', ' ', text)
    # Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Hapus simbol non-alfanumerik (jika masih ada)
    text = re.sub(r'[^\w\s]', ' ', text)
    # Normalisasi kata
    text = normalize_text(text)
    # Trim spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned'] = df['Komentar'].apply(clean_text)

In [ ]:
# 3.3 Tokenisasi sederhana (split by whitespace)
df['tokens'] = df['cleaned'].apply(lambda x: x.split())

In [ ]:
# 3.4 Stopword removal (Sastrawi)
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())
df['stopwords_removed'] = df['tokens'].apply(lambda toks: [w for w in toks if w not in stopwords])

In [ ]:
# 3.5 Stemming (Sastrawi) - stem tiap kata unik sekali lalu mapping untuk efisiensi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# kumpulkan kata unik
all_words = set(w for toks in df['stopwords_removed'] for w in toks)
print("Kata unik sebelum stemming:", len(all_words))

word_map = {}
for w in tqdm(all_words, desc="Stemming kata unik"):
    word_map[w] = stemmer.stem(w)

# Terapkan mapping ke setiap token list
df['stemming'] = df['stopwords_removed'].apply(lambda toks: [word_map[w] for w in toks])

In [ ]:
# 3.5 Gabungkan kembali menjadi string untuk representasi
df['clean_text'] = df['stemming'].apply(lambda toks: " ".join(toks))

display(df[['Komentar', 'cleaned','tokens', 'stopwords_removed', 'stemming', 'clean_text']].head(5))

# **4. Labelling**

In [ ]:
# install library
!pip install -q transformers

**4.1 Load Pretrained IndoBERT Sentiment Model (binary POS/NEG)**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

SENTIMENT_MODEL_NAME = "agufsamudra/indo-sentiment-analysis"

tokenizer_sent = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_NAME)
model_sent = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_NAME)

device = 0 if torch.cuda.is_available() else -1

sentiment_clf = pipeline(
    "text-classification",
    model=model_sent,
    tokenizer=tokenizer_sent,
    device=device
)

texts = df["clean_text"].tolist()
print("Jumlah komentar yang akan diprediksi:", len(texts))

# Pipeline memproses list teks (otomatis batch)
results = sentiment_clf(texts, batch_size=32, truncation=True)

# Simpan label mentah & skor confidence
df["sentiment_model_label_raw"] = [r["label"] for r in results]
df["sentiment_model_score"] = [r["score"] for r in results]

print("\nContoh hasil prediksi model (5 baris):")
display(
    df[
        ["clean_text", "sentiment_model_label_raw", "sentiment_model_score"]
    ].head()
)

**4.2 Mapping label model: positif/negatif**

In [ ]:
# Label unknown akan di drop unknown untuk modeling, balancing, dan encoding

def map_model_label_to_sentiment(label_str: str) -> str:
    l = label_str.lower()
    if "neg" in l or "0" in l:
        return "negatif"
    if "pos" in l or "1" in l:
        return "positif"
    return "unknown"

CONFIDENCE_THRESHOLD = 0.6

def final_sentiment_label(row):
    base = map_model_label_to_sentiment(row["sentiment_model_label_raw"])
    score = row["sentiment_model_score"]

    if score < CONFIDENCE_THRESHOLD:
        return "unknown"
    return base

df["sentiment_label"] = df.apply(final_sentiment_label, axis=1)

print("Distribusi label (termasuk unknown):")
print(df["sentiment_label"].value_counts())

In [ ]:
NO_FEEDBACK_KEYWORDS = ["masukan", "saran", "kritik", "kiritik", "komentar"]

def adjust_no_feedback_labels(row):
    t = row["clean_text"].lower()
    label = row["sentiment_label"]

    # jika mengandung "tidak ada + kata kunci"
    if "tidak ada" in t and any(k in t for k in NO_FEEDBACK_KEYWORDS):

        # jika ada kata positif → jadikan positif
        if "bagus" in t or "baik" in t or "cukup" in t:
            return "positif"

        # selain itu → tidak jelas → unknown
        return "unknown"

    return label

df["sentiment_label"] = df.apply(adjust_no_feedback_labels, axis=1)

print("Distribusi label setelah penyesuaian rule-based:")
print(df["sentiment_label"].value_counts())

**4.3 Filter hanya positif & negatif, balancing, dan encoding**

In [ ]:
# Hanya gunakan data dengan label positif & negatif
df_labeled = df[df["sentiment_label"].isin(["positif", "negatif"])].copy()

print("Distribusi label setelah mengeluarkan 'unknown':")
print(df_labeled["sentiment_label"].value_counts())

# Balancing sederhana: undersampling kelas mayoritas
class_counts = df_labeled["sentiment_label"].value_counts()
min_count = class_counts.min()

print("\nJumlah data tiap kelas sebelum balancing:")
for label, count in class_counts.items():
    print(f" - {label}: {count}")

print("\nUkuran target tiap kelas setelah balancing (undersampling):", min_count)

dfs_balanced = []
for label in class_counts.index:
    sample = df_labeled[df_labeled["sentiment_label"] == label].sample(
        n=min_count, random_state=42
    )
    dfs_balanced.append(sample)

df_balanced = pd.concat(dfs_balanced).sample(frac=1, random_state=42).reset_index(drop=True)
print("\nDistribusi label setelah balancing:")
print(df_balanced["sentiment_label"].value_counts())

# Konversi label ke bentuk numerik (0/1) untuk modeling
# negatif = 0, positif = 1
label_mapping = {"negatif": 0, "positif": 1}
df_balanced["label"] = df_balanced["sentiment_label"].map(label_mapping)

print("\nContoh 10 baris data siap modeling:")
display(
    df_balanced[
        ["Komentar", "clean_text", "sentiment_model_label_raw",
         "sentiment_model_score", "sentiment_label", "label"]
    ].head(10)
)

**3.5 Simpan Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/grive')

# Simpan dataset hasil cleaning & labeling (file CSV)
OUTPUT_PATH = "/content/grive/My Drive/1.KULIAH/TA/data2/analisis/edom-2024-clean-balanced.csv"

# Ensure the directory exists before saving
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df_balanced.to_csv(OUTPUT_PATH, index=False)
print("Dataset cleaned & labeled tersimpan di:", OUTPUT_PATH)

In [ ]:
# Simpan dataset hasil cleaning & labeling (file EXCEL)
OUTPUT_XLSX = "/content/grive/My Drive/1.KULIAH/TA/data2/analisis/edom-2024-clean-balanced.xlsx"

# Ensure the directory exists before saving
import os
os.makedirs(os.path.dirname(OUTPUT_XLSX), exist_ok=True)

df_balanced.to_excel(OUTPUT_XLSX, index=False)
print("Dataset cleaned & labeled tersimpan di:", OUTPUT_XLSX)

# **INDOBERT + KNN VS TF-IDF + KNN**

In [ ]:
import pandas as pd
from IPython.display import display

# load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load dataset hasil preparation
DATASET_PATH = "/content/drive/MyDrive/1.KULIAH/TA/data2/analisis/edom-2024-clean-balanced.csv"

# Membaca dataset
df = pd.read_csv(DATASET_PATH)

print("Ukuran dataset:", df.shape)
print("\nNama kolom:")
print(df.columns.tolist())

print("\nLima baris pertama dataset:")
display(df.head())

# Kolom yang akan digunakan untuk modeling
TEXT_COLUMN = "clean_text"   # fitur teks
LABEL_COLUMN = "label"             # target numerik (0/1)

# Cek apakah kolom ada
for col in [TEXT_COLUMN, LABEL_COLUMN]:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan di dataset. Sesuaikan nama kolomnya.")

# **Mengubah Kolom Dosen Menjadi Anonym**

In [ ]:
def anonymize_column(df, column_name, prefix):
    """
    Mengubah isi kolom menjadi samaran konsisten.
    Contoh:
    Ahmad Fauzi → Dosen_01
    Ahmad Fauzi (muncul lagi) → Dosen_01
    """

    unique_values = df[column_name].dropna().unique()

    mapping = {
        name: f"{str(i+1).zfill(2)}"
        for i, name in enumerate(unique_values)
    }

    df[column_name] = df[column_name].map(mapping)

    return df, mapping

df, dosen_mapping = anonymize_column(
    df,
    column_name="Dosen",
    prefix="Nama_Dosen"
)

df.head()

# **Split Data**

In [ ]:
from sklearn.model_selection import train_test_split

# Fitur teks dan target
X_text = df[TEXT_COLUMN].astype(str).tolist()
y = df[LABEL_COLUMN].astype(int)

# Bagi data train dan test (80:20)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # supaya proporsi kelas seimbang di train & test
)

print("Jumlah data training :", len(X_train_text))
print("Jumlah data testing  :", len(X_test_text))

# **Representasi TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# === TF-IDF TRAINING ===
tfidf = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,              # kata yang muncul min 2x
    max_df=0.9             # abaikan kata yang terlalu umum
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF shape train:", X_train_tfidf.shape, "test:", X_test_tfidf.shape)

# Tampilkan matriks TF-IDF untuk 4 baris pertama
#df_tfidf_preview = pd.DataFrame(X_train_tfidf[:4].toarray(),columns=tfidf.get_feature_names_out())

#print("\nContoh 4 baris pertama matriks TF-IDF:")
#display(df_tfidf_preview.head())

# === TABEL TF-IDF PER KOLOM UNTUK BARIS 1-3 ===
df_full = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())

# Pilih baris 1, 2, 3 (index 0-2)
tfidf_rows_subset = df_full.iloc[:3]

print("\n=== Nilai TF-IDF Non-Nol untuk Baris 1, 2, dan 3 ===")

for index, row in tfidf_rows_subset.iterrows():
    non_zero_values = row[row != 0.0]

    print(f"\nBaris ke-{index + 1}:")
    if not non_zero_values.empty:
        tfidf_table = non_zero_values.reset_index()
        tfidf_table.columns = ['Kata', 'Hasil Perhitungan TF-IDF']
        print(tfidf_table.to_markdown(index=False, floatfmt=".4f"))
    else:
        print("Tidak ada nilai TF-IDF non-nol.")

## **Persebaran Term Frequency (TF)**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Inisialisasi CountVectorizer
count_vectorizer = CountVectorizer()

# Transform data latih
X_train_tf = count_vectorizer.fit_transform(X_train_text)

# Total frekuensi kata di seluruh dokumen (TF mentah)
tf_sum = X_train_tf.sum(axis=0).A1

# Total kata per dokumen
total_terms_per_doc = X_train_tf.sum(axis=1)

# Hitung TF ternormalisasi per dokumen
tf_normalized = X_train_tf.multiply(1 / total_terms_per_doc)

# Rata-rata TF per kata di seluruh dokumen
tf_score = tf_normalized.mean(axis=0).A1

# Membuat DataFrame
df_tf = pd.DataFrame({
    "Kata": count_vectorizer.get_feature_names_out(),
    "Total_Term_Frequency": tf_sum,
    "Skor_TF": tf_score
}).sort_values("Total_Term_Frequency", ascending=False)

# Menampilkan 15 kata teratas
df_tf.head(15)

## **Persebaran Inverse Document Frequency (IDF)**

In [ ]:
# Mengambil nilai IDF dari objek TF-IDF Vectorizer
idf_values = tfidf.idf_

# Membuat DataFrame
df_idf = pd.DataFrame({
    "Kata": tfidf.get_feature_names_out(),
    "Skor_IDF": idf_values,
    "Jumlah_Dokumen (DF)": (X_train_tfidf > 0).sum(axis=0).A1
})

# SORT BERDASARKAN SKOR IDF TERTINGGI
df_idf_tertinggi = df_idf.sort_values(
    "Skor_IDF", ascending=False
)

# Tampilkan 15 kata dengan IDF tertinggi
df_idf_tertinggi.head(15)

## **Total TF-IDF**

In [ ]:
idx = 0

# 1. Ambil teks asli dari X_train (bukan dari df awal agar tidak tertukar indeksnya)
teks_asli = X_train_text[idx]

# 2. Ambil baris TF-IDF yang sesuai
sample_tfidf_row = X_train_tfidf[idx]

# 3. Buat DataFrame bobot
df_sample = pd.DataFrame({
    "Kata": tfidf.get_feature_names_out(),
    "Bobot_TF-IDF": sample_tfidf_row.toarray().flatten()
})

# 4. Filter dan urutkan
df_sample = df_sample[df_sample["Bobot_TF-IDF"] > 0].sort_values("Bobot_TF-IDF", ascending=False)

print(f"Komentar Riil di Indeks {idx}:\n'{teks_asli}'")
print("\nBobot TF-IDF yang Terdeteksi:")
print(df_sample.to_string(index=False))

## **WordCloud Perhitungan TF-IDF**

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# === HITUNG TOTAL BOBOT TF-IDF ===
tfidf_scores = X_train_tfidf.sum(axis=0).A1

tfidf_dict = dict(
    zip(tfidf.get_feature_names_out(), tfidf_scores)
)

# === WORDCLOUD ===
wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color='white',
    max_words=100,
    colormap='viridis'
).generate_from_frequencies(tfidf_dict)

plt.figure(figsize=(15,6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud Berdasarkan Bobot TF-IDF", fontsize=14)
plt.show()

# **Representasi IndoBERT**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print("Loaded IndoBERT on", device)

In [ ]:
def embed_texts_batched(texts, batch_size=32, max_length=128, token_sample_n=2):
    pooled = []
    # token_tables: menyimpan tabel token-level untuk beberapa sampel (hanya untuk ditampilkan)
    token_tables = {}

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Tokenisasi batch (padding + truncation)
        enc = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(device)

        # Non-gradient karena hanya inference
        with torch.no_grad():
            out = model(**enc)

         # Ambil hidden state terakhir (B, seq_len, hidden_dim)
        last_hidden = out.last_hidden_state.cpu().numpy()  # shape (B, seq_len, hidden)

        # Attention mask untuk tahu posisi token asli (bukan padding)
        masks = enc['attention_mask'].cpu().numpy()
        for b_idx in range(len(batch)):
            seq = last_hidden[b_idx]
            mask = masks[b_idx]

            if mask.sum() > 0:
                pooled_vec = (seq * mask[:, None]).sum(axis=0) / mask.sum()
            else:
                pooled_vec = seq.mean(axis=0)
            pooled.append(pooled_vec)

        # Buat token-level table (hanya untuk sampel pertama batch 1)
        # token_sample_n menentukan berapa sample yang ditampilkan
        if i == 0:
            for j in range(min(token_sample_n, len(batch))):
                token_ids = enc['input_ids'][j].cpu().numpy()
                tokens = tokenizer.convert_ids_to_tokens(token_ids)
                seq_j = last_hidden[j]
                dims_show = min(8, seq_j.shape[1])

                data = []
                for t_i, tok in enumerate(tokens):
                    row = [tok] + list(seq_j[t_i,:dims_show]) + [float(np.linalg.norm(seq_j[t_i]))]
                    data.append(row)

                cols = ['token'] + [f'dim_{d}' for d in range(dims_show)] + ['l2_norm']
                token_tables[j] = pd.DataFrame(data, columns=cols)
    return np.vstack(pooled), token_tables

# ============================================================
#          JALANKAN PROSES EMBEDDING TRAIN & TEST
# ============================================================
# --------------- EMBEDDING TRAIN ---------------
start = time.time()
X_train_bert, token_tables_train = embed_texts_batched(X_train_text, batch_size=32, token_sample_n=2)
end_train = time.time()
print(f"Embedding for train data done in {end_train-start:.1f}s; shape:", X_train_bert.shape)

# --------------- EMBEDDING TEST ---------------
start = time.time()
X_test_bert, _ = embed_texts_batched(X_test_text, batch_size=32, token_sample_n=0) # token_sample_n=0 to avoid re-generating tables
end_test = time.time()
print(f"Embedding for test data done in {end_test-start:.1f}s; shape:", X_test_bert.shape)

# ============================================================
#      TAMPILKAN TOKEN-LEVEL TABLE (2 SAMPLE PERTAMA)
# ============================================================
for k,v in token_tables_train.items():
    print(f"\nToken-level table sample {k} (from train data):")
    display(v.head(10))

## **Contoh Embedding 1 Komentar**

In [ ]:
#Ambil 1 contoh komentar
sample_text = X_train_text[1]
print("Komentar contoh:")
print(sample_text)

#Tokenisasi & embedding satu komentar
enc = tokenizer(
    sample_text,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=128
).to(device)

with torch.no_grad():
    out = model(**enc)

last_hidden = out.last_hidden_state.cpu().numpy()[0]
mask = enc['attention_mask'].cpu().numpy()[0]
tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])

#Tabel hasil Embedding
rows = []
for i, tok in enumerate(tokens):
    if mask[i] == 1:
        rows.append({
            "Token": tok,
            "Contoh Nilai (3 dimensi)": last_hidden[i][:3],
            "Panjang Vektor (L2 Norm)": np.linalg.norm(last_hidden[i])
        })

df_explain = pd.DataFrame(rows)
display(df_explain)

#Hitung Embedding Kalimat(Rata-rata)
sentence_embedding = (last_hidden * mask[:, None]).sum(axis=0) / mask.sum()

print("Embedding kalimat (5 dimensi pertama):")
print(sentence_embedding[:5])
print("\nPanjang vektor embedding kalimat:", np.linalg.norm(sentence_embedding))

# **Evaluasi Model**

## **TF-IDF + KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# ======================================================
# Fungsi: Mencari nilai K terbaik memakai Cross-Validation
# ======================================================
def select_best_k_cv(X_train, y_train, k_list, cv=5):
    best_k = None
    best_score = -1
    scores_mean = {}

    print("=== Evaluasi KNN (Cross-Validation f1_macro) ===")

    for k in k_list:
        model = KNeighborsClassifier(n_neighbors=k)

        scores = cross_val_score(
            model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1
        )

        mean_score = scores.mean()
        scores_mean[k] = mean_score

        print(f"K = {k:2d} | mean f1_macro = {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_k = k

    print("\n=== K TERBAIK (Cross-Validation) ===")
    print(f"Best K  : {best_k}")
    print(f"Best CV f1_macro : {best_score:.4f}")

    return best_k, scores_mean


# ======================================================
# 1. Daftar nilai K (ambil hanya yang ganjil)
# ======================================================
k_list = list(range(1, 32, 2))  # K = 1,3,5,...31

# ======================================================
# 2. Jalankan untuk TF-IDF
# ======================================================
best_k_tfidf, scores_tfidf = select_best_k_cv(
    X_train_tfidf, y_train, k_list)

print("\nBest K TF-IDF:", best_k_tfidf)

# =============================
# 3. Plot hasil CV untuk visualisasi
# =============================
plt.figure(figsize=(8, 5))
plt.plot(list(scores_tfidf.keys()), list(scores_tfidf.values()), marker='o')
plt.xlabel("Nilai K")
plt.ylabel("F1 Macro (CV Mean)")
plt.title("Cross-Validation F1-Macro vs K (TF-IDF)")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# EVALUASI KNN + TF-IDF (PAKAI K TERBAIK CV)
# ==========================================

# Inisialisasi KNN dengan K terbaik hasil CV
knn_tfidf = KNeighborsClassifier(n_neighbors=best_k_tfidf)

# Training model
knn_tfidf.fit(X_train_tfidf, y_train)

# Prediksi
pred_tfidf = knn_tfidf.predict(X_test_tfidf)

# ====== METRIK ======
print("TF-IDF + KNN — Accuracy:", accuracy_score(y_test, pred_tfidf))
print("F1 macro:", f1_score(y_test, pred_tfidf, average='macro'))

print("\n=== Classification Report (TF-IDF + KNN) ===")
print(classification_report(y_test, pred_tfidf, target_names=['neg', 'pos']))

# ====== CONFUSION MATRIX ======
cm = confusion_matrix(y_test, pred_tfidf)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['neg', 'pos'],
    yticklabels=['neg', 'pos']
)
plt.title(f'TF-IDF + KNN (K={best_k_tfidf}) - Confusion Matrix')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## **IndoBERT + KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

# ======================================================
# Fungsi: Mencari K terbaik menggunakan Cross-Validation
# ======================================================
def get_best_k_cv(X_train, y_train, k_values, cv=5):
    k_scores = {}
    best_k = None
    best_score = -1

    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k)
        scores = cross_val_score(
            model, X_train, y_train,
            cv=cv,
            scoring='f1_macro',
            n_jobs=-1
        )

        mean_score = scores.mean()
        k_scores[k] = mean_score

        print(f"K={k} | f1_macro CV mean: {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_k = k

    return best_k, k_scores


# =============================
# 1. Tentukan daftar nilai K
# =============================
k_values = list(range(1, 32, 2))  # hanya K ganjil, 1–31

# =============================
# 2. Jalankan pencarian K terbaik
# =============================
best_k_bert, score_dict_bert = get_best_k_cv(
    X_train_bert, y_train, k_values
)

print("\n=== HASIL K TERBAIK INDO BERT (CV) ===")
print("Best K:", best_k_bert)
print("F1-macro terbaik:", score_dict_bert[best_k_bert])

# =============================
# 3. Plot hasil CV untuk visualisasi
# =============================
plt.figure(figsize=(8, 5))
plt.plot(list(score_dict_bert.keys()), list(score_dict_bert.values()), marker='o')
plt.xlabel("Nilai K")
plt.ylabel("F1 Macro (CV Mean)")
plt.title("Cross-Validation F1-Macro vs K (IndoBERT Embeddings)")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# EVALUASI KNN + INDOBERT (PAKAI K TERBAIK CV)
# ==========================================

knn_bert = KNeighborsClassifier(n_neighbors=best_k_bert)

# train model
knn_bert.fit(X_train_bert, y_train)

# prediksi pada data test
pred_bert = knn_bert.predict(X_test_bert)

# ===== METRIK =====
print("IndoBERT + KNN — Accuracy:", accuracy_score(y_test, pred_bert))
print("F1 macro:", f1_score(y_test, pred_bert, average='macro'))

print("\n=== Classification Report (IndoBERT + KNN) ===")
print(classification_report(y_test, pred_bert, target_names=['neg', 'pos']))

# ===== CONFUSION MATRIX =====
cm = confusion_matrix(y_test, pred_bert)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['neg', 'pos'],
            yticklabels=['neg', 'pos'])
plt.title(f'IndoBERT + KNN (K={best_k_bert}) - Confusion Matrix')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# **Perbandingan Hasil**

In [ ]:
def metrics_for(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro')
    }

results = {
    'TF-IDF + KNN': metrics_for(y_test, pred_tfidf),
    'IndoBERT + KNN': metrics_for(y_test, pred_bert),
}

results_df = pd.DataFrame(results).T
display(results_df)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

def metrics_full(y_true, y_pred):
    """
    Menghasilkan metrik evaluasi lengkap:
    - Accuracy
    - Macro Precision, Recall, F1
    - Precision, Recall, F1 per kelas (negatif & positif)
    """

    report = classification_report(
        y_true,
        y_pred,
        target_names=['neg', 'pos'],
        output_dict=True
    )

    return {
        # ============================
        # METRIK GLOBAL
        # ============================
        'accuracy': accuracy_score(y_true, y_pred),

        # ============================
        # METRIK MACRO
        # ============================
        'precision_macro': report['macro avg']['precision'],
        'recall_macro': report['macro avg']['recall'],
        'f1_macro': report['macro avg']['f1-score'],

        # ============================
        # KELAS NEGATIF
        # ============================
        'precision_neg': report['neg']['precision'],
        'recall_neg': report['neg']['recall'],
        'f1_neg': report['neg']['f1-score'],

        # ============================
        # KELAS POSITIF
        # ============================
        'precision_pos': report['pos']['precision'],
        'recall_pos': report['pos']['recall'],
        'f1_pos': report['pos']['f1-score'],
    }

results = {
    'TF-IDF + KNN': metrics_full(y_test, pred_tfidf),
    'IndoBERT + KNN': metrics_full(y_test, pred_bert),
}

results_df = pd.DataFrame(results).T
display(results_df.round(3))

# **Simpan Model**

In [ ]:
import joblib
import os

# ============================================================
# Folder tujuan penyimpanan model (Google Drive)
# ============================================================
MODEL_DIR = "/content/drive/MyDrive/1.KULIAH/TA/data2/analisis/model"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Folder penyimpanan model:", MODEL_DIR)

# ============================================================
# Simpan model KNN IndoBERT
# ============================================================
MODEL_PATH = os.path.join(MODEL_DIR, "knn_indobert.pkl")

joblib.dump(knn_bert, MODEL_PATH)

print("Model KNN IndoBERT berhasil disimpan di:")
print(MODEL_PATH)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import os

MODEL_NAME = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

SAVE_DIR = "/content/drive/MyDrive/1.KULIAH/TA/data2/analisis/model/indobert"
os.makedirs(SAVE_DIR, exist_ok=True)

tokenizer.save_pretrained(SAVE_DIR)
model.save_pretrained(SAVE_DIR)

print("IndoBERT tokenizer & model berhasil disimpan di Drive:")
print(SAVE_DIR)